# M4A_01: Temporal Features for Commuter Prediction

**Track A: Binary Classification (15-Minute Availability)**

## 🎯 Learning Objectives
By the end of this notebook, you will:
- Extract time-based features from timestamps
- Create rush hour indicators for commuter patterns
- Engineer cyclical time encodings (sin/cos transformations)
- Identify peak demand periods through visualization
- Prepare temporal features for classification models

## 📊 Why Temporal Features Matter
Commuter behavior follows **strong time patterns**:
- Morning rush: 6-9 AM (bikes depleting)
- Evening rush: 4-7 PM (bikes returning)
- Weekend patterns differ from weekdays
- Holidays affect demand

**Our Goal**: Capture these patterns as model features.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

Load the merged bike + weather dataset from Module 2.

In [ ]:
# TODO: Load data from data/processed/ or data/raw/
# Hint: You should have a merged dataset from Module 2
df = pd.read_csv('../../../data/raw/sample_bike_weather.csv')

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset shape: {df.shape}")
df.head()

## 2. Basic Time Features

Extract hour, day of week, month, and year from timestamps.

In [ ]:
# Extract basic time components
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek  # Monday=0, Sunday=6
df['day_name'] = df['timestamp'].dt.day_name()
df['month'] = df['timestamp'].dt.month
df['year'] = df['timestamp'].dt.year
df['date'] = df['timestamp'].dt.date

print("Time features created:")
df[['timestamp', 'hour', 'day_of_week', 'day_name', 'month']].head(10)

## 3. Rush Hour Indicators

Create binary features for commuter rush hours:
- **Morning rush**: 6-9 AM
- **Evening rush**: 4-7 PM (16-19 in 24-hour format)

In [ ]:
# Morning rush hour (6-9 AM)
df['is_morning_rush'] = ((df['hour'] >= 6) & (df['hour'] < 9)).astype(int)

# Evening rush hour (4-7 PM)
df['is_evening_rush'] = ((df['hour'] >= 16) & (df['hour'] < 19)).astype(int)

# Any rush hour
df['is_rush_hour'] = ((df['is_morning_rush'] == 1) | (df['is_evening_rush'] == 1)).astype(int)

# Weekday indicator (Monday-Friday)
df['is_weekday'] = (df['day_of_week'] < 5).astype(int)

# Weekend indicator
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

print("Rush hour distribution:")
print(df[['is_morning_rush', 'is_evening_rush', 'is_rush_hour']].sum())
print(f"\nWeekday records: {df['is_weekday'].sum()}")
print(f"Weekend records: {df['is_weekend'].sum()}")

### 📊 Visualize Rush Hour Patterns

In [ ]:
# Average bike availability by hour
hourly_avg = df.groupby('hour')['bikes_available'].mean()

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.bar(hourly_avg.index, hourly_avg.values, color='steelblue', alpha=0.7)
plt.axvspan(6, 9, alpha=0.2, color='red', label='Morning Rush')
plt.axvspan(16, 19, alpha=0.2, color='orange', label='Evening Rush')
plt.xlabel('Hour of Day')
plt.ylabel('Average Bikes Available')
plt.title('Bike Availability by Hour (Rush Hours Highlighted)')
plt.legend()
plt.grid(axis='y', alpha=0.3)

# Weekday vs Weekend comparison
plt.subplot(1, 2, 2)
weekday_hourly = df[df['is_weekday'] == 1].groupby('hour')['bikes_available'].mean()
weekend_hourly = df[df['is_weekend'] == 1].groupby('hour')['bikes_available'].mean()

plt.plot(weekday_hourly.index, weekday_hourly.values, marker='o', label='Weekday', linewidth=2)
plt.plot(weekend_hourly.index, weekend_hourly.values, marker='s', label='Weekend', linewidth=2)
plt.xlabel('Hour of Day')
plt.ylabel('Average Bikes Available')
plt.title('Weekday vs Weekend Patterns')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Cyclical Time Encodings

**Problem**: Hour 23 and hour 0 are close in time, but numerically far apart (23 vs 0).  
**Solution**: Use sin/cos transformations to create cyclical features.

### Mathematical Intuition
- Hour 0 and Hour 23 should be "close" (both late night)
- Hour 6 and Hour 18 should be "opposite" (morning vs evening)
- Sin/cos transformation maps linear time to a circle

In [ ]:
# Hour cyclical encoding (24 hours)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Day of week cyclical encoding (7 days)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# TODO: Create month cyclical encoding (12 months)
# Hint: Follow the same pattern as above
# df['month_sin'] = ...
# df['month_cos'] = ...

print("Cyclical features created:")
df[['hour', 'hour_sin', 'hour_cos', 'day_of_week', 'day_of_week_sin', 'day_of_week_cos']].head(10)

### 📊 Visualize Cyclical Encoding

In [ ]:
# Plot hour encoding on unit circle
plt.figure(figsize=(8, 8))
unique_hours = df[['hour', 'hour_sin', 'hour_cos']].drop_duplicates().sort_values('hour')

plt.scatter(unique_hours['hour_cos'], unique_hours['hour_sin'], 
           c=unique_hours['hour'], cmap='twilight', s=200, edgecolors='black', linewidth=2)
plt.colorbar(label='Hour of Day')

# Annotate key hours
for _, row in unique_hours.iterrows():
    if row['hour'] % 6 == 0:  # Annotate every 6 hours
        plt.annotate(f"{int(row['hour'])}h", 
                    (row['hour_cos'], row['hour_sin']),
                    fontsize=12, fontweight='bold',
                    xytext=(5, 5), textcoords='offset points')

plt.xlabel('Hour Cosine')
plt.ylabel('Hour Sine')
plt.title('Cyclical Encoding of Hour on Unit Circle')
plt.axhline(0, color='gray', linewidth=0.5, linestyle='--')
plt.axvline(0, color='gray', linewidth=0.5, linestyle='--')
plt.grid(alpha=0.3)
plt.axis('equal')
plt.show()

## 5. Holiday Indicators (Optional Enhancement)

If you have holiday data, create binary indicators for Dutch public holidays.

In [ ]:
# Example: Manually define Dutch public holidays for 2024
dutch_holidays_2024 = [
    '2024-01-01',  # New Year's Day
    '2024-03-29',  # Good Friday
    '2024-03-31',  # Easter Sunday
    '2024-04-01',  # Easter Monday
    '2024-04-27',  # King's Day
    '2024-05-05',  # Liberation Day
    '2024-05-09',  # Ascension Day
    '2024-05-19',  # Whit Sunday
    '2024-05-20',  # Whit Monday
    '2024-12-25',  # Christmas Day
    '2024-12-26',  # Boxing Day
]

# Convert to datetime
dutch_holidays_2024 = pd.to_datetime(dutch_holidays_2024)

# Create holiday indicator
df['is_holiday'] = df['date'].isin(dutch_holidays_2024.date).astype(int)

print(f"Holiday records: {df['is_holiday'].sum()}")
print(f"\nHoliday dates in dataset:")
print(df[df['is_holiday'] == 1][['timestamp', 'date']].drop_duplicates())

## 6. Summary: Temporal Features Created

Review all temporal features for Track A classification.

In [ ]:
# List all temporal features
temporal_features = [
    'hour', 'day_of_week', 'month', 'year',
    'is_morning_rush', 'is_evening_rush', 'is_rush_hour',
    'is_weekday', 'is_weekend',
    'hour_sin', 'hour_cos',
    'day_of_week_sin', 'day_of_week_cos',
    'is_holiday'
]

print("Temporal features for Track A:")
print(df[temporal_features].describe())

# Save feature-engineered dataset
# df.to_csv('../../../data/processed/track_a_temporal_features.csv', index=False)
# print("\n✅ Features saved to data/processed/track_a_temporal_features.csv")

## 🎯 Key Takeaways

1. **Rush hour indicators** capture peak commuter demand periods
2. **Cyclical encodings** (sin/cos) handle the circular nature of time
3. **Weekday/weekend flags** distinguish commuter vs leisure patterns
4. **Holiday indicators** capture special days with different demand

## 🔗 Next Steps
- **M4A_02**: Weather features (current conditions)
- **M4A_03**: Train schedule features (next 15-30 minutes)
- **Module 5 Track A**: Classification models